In [1]:
# --- Import all necessary libraries ---
import os
import sys
import json
import asyncio
import random
import string
from uuid import uuid4
from typing import Any, List
from dotenv import load_dotenv
import pandas as pd
# import plotly.graph_objects as go
from IPython.display import HTML, Markdown, display

# --- ADK, Agent, and Evaluation Components ---
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content, Part



load_dotenv()

True

In [ ]:
from google import genai
from google.adk import Agent

root_agent = Agent(
    name="greeting_agent",
    model="gemini-2.5-flash",
    instruction="You are a helpful assistant. You have to reply in single word only",
)

In [3]:
root_agent.name

'greeting_agent'

In [6]:
from google.adk.runners import InMemoryRunner

runner = InMemoryRunner(agent=root_agent)

response = await runner.run_debug(
    "what comes after monday",
    verbose=True,
)

greeting_agent > Tuesday


In [7]:
response

[Event(model_version='gemini-2.5-flash', content=Content(
   parts=[
     Part(
       text='Tuesday'
     ),
   ],
   role='model'
 ), grounding_metadata=None, partial=None, turn_complete=None, turn_complete_reason=None, finish_reason=<FinishReason.STOP: 'STOP'>, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=GenerateContentResponseUsageMetadata(
   candidates_token_count=1,
   prompt_token_count=35,
   prompt_tokens_details=[
     ModalityTokenCount(
       modality=<MediaModality.TEXT: 'TEXT'>,
       token_count=35
     ),
   ],
   total_token_count=36
 ), live_session_resumption_update=None, live_session_id=None, go_away=None, input_transcription=None, output_transcription=None, avg_logprobs=None, logprobs_result=None, cache_metadata=None, citation_metadata=None, interaction_id=None, invocation_id='e-4e7679f1-5938-4eb9-86c9-224fe6cd9b89', author='greeting_agent', actions=EventActions(skip_summarization=None, state_delta={}, artifact_del

In [8]:
from google.genai import types
from google.adk.agents import Agent
from google.adk.models.google_llm import Gemini
from google.adk.tools import google_search

retry_config=types.HttpRetryOptions(
    attempts=5,         # Maximum retry attempts
    exp_base=7,         # Delay multiplier
    initial_delay=1,    # Initial delay before first retry (in seconds)
    http_status_codes=[
        429, # Too Many Requests
        500, # Internal Server Error
        503, # Service Unavailable
        504, # Gateway Timeout
        ] # Retry on these HTTP errors
)

root_agent = Agent(
    name="assistant",
    model=Gemini(
        model="gemini-2.5-flash",
        retry_options=retry_config
    ),
    description="A simple agent that can answer general questions.",
    instruction="""You are a helpful assistant.
    Use Google Search for current info or if unsure. reply back with one word answer only""",
    tools=[google_search],
)

In [9]:
from google.adk.runners import InMemoryRunner
runner = InMemoryRunner(agent=root_agent)

In [ ]:
from google.adk.runners import InMemoryRunner
runner = InMemoryRunner(agent=root_agent)

response = await runner.run_debug(
    "how much price increased in mac book pro recently?",
    verbose=True,
)

assistant > Apple recently increased the prices of its MacBook Pro models, citing a memory chip shortage driven by demand from AI data centers. These price adjustments were announced around June 26, 2026.

Specific increases for MacBook Pro models include:
*   The entry-level 14-inch MacBook Pro (M5 model) increased from $1,599 to $1,999, a $400 hike.
*   The one-terabyte MacBook Pro (M5 model) is now priced at $1,999, up from $1,699.
*   The 14-inch MacBook Pro with the M5 Pro chip saw a $300 increase, now starting at $2,500.
*   The 16-inch MacBook Pro (M5 Pro) now begins at $3,000, a $500 increase from its previous starting price of $2,499. However, this base model now includes 1TB of storage, double the previous 512GB.

Beyond base models, upgrade options for RAM and SSDs also saw significant price jumps, with RAM upgrades increasing by up to 100% and SSD upgrades by 50-67% in most configurations. This means that a fully equipped 16-inch MacBook Pro can now cost over $10,000.


In [11]:
response

[Event(model_version='gemini-2.5-flash', content=Content(
   parts=[
     Part(
       text="""Apple recently increased the prices of its MacBook Pro models, citing a memory chip shortage driven by demand from AI data centers. These price adjustments were announced around June 26, 2026.
 
 Specific increases for MacBook Pro models include:
 *   The entry-level 14-inch MacBook Pro (M5 model) increased from $1,599 to $1,999, a $400 hike.
 *   The one-terabyte MacBook Pro (M5 model) is now priced at $1,999, up from $1,699.
 *   The 14-inch MacBook Pro with the M5 Pro chip saw a $300 increase, now starting at $2,500.
 *   The 16-inch MacBook Pro (M5 Pro) now begins at $3,000, a $500 increase from its previous starting price of $2,499. However, this base model now includes 1TB of storage, double the previous 512GB.
 
 Beyond base models, upgrade options for RAM and SSDs also saw significant price jumps, with RAM upgrades increasing by up to 100% and SSD upgrades by 50-67% in most configurat

## memory

In [13]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

# Set up Session Management
session_service = InMemorySessionService()

# Set up the agent
root_agent = Agent(
    name="assistant",
    model=Gemini(
        model="gemini-2.5-flash",
        retry_options=retry_config
    ),
    description="A simple agent that can answer general questions.",
    instruction="""You are a helpful assistant. answer in one word"""
)

# Create the Runner
runner = Runner(
    agent=root_agent,
    app_name="default",
    session_service=session_service
    )

In [14]:
session_name =  "session1"
app_name = runner.app_name
USER_ID = 'default'

# Create a new session
session = await session_service.create_session(
    app_name=app_name,
    user_id=USER_ID,
    session_id=session_name
)

user_queries = [
    "Hi, I am Sam! What is the capital of United States?",
    "Hello! What is my name?",
]

for query in user_queries:
    print(f"\nUser > {query}")

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session.id,
        new_message=types.Content(role="user", parts=[types.Part(text=query)])
    ):
      print(f"Assistant > ", event.content.parts[0].text)


User > Hi, I am Sam! What is the capital of United States?
Assistant >  Washington

User > Hello! What is my name?
Assistant >  Sam


In [16]:
from google.genai import types
from google.genai.types import Content, Part

Content(role="user", parts=[Part(text=query)])

Content(
  parts=[
    Part(
      text='Hello! What is my name?'
    ),
  ],
  role='user'
)

In [17]:
session = await session_service.get_session(
    app_name=app_name,
    user_id=USER_ID,
    session_id=session_name,
)

for event in session.events:
    print(f"{event.content.role}: {event.content.parts[0].text}")

user: Hi, I am Sam! What is the capital of United States?
model: Washington
user: Hello! What is my name?
model: Sam


In [ ]:
from google.adk.sessions import DatabaseSessionService

db_url = "sqlite:///my_agent_data.db"  # Local SQLite file
session_service = DatabaseSessionService(db_url=db_url)

In [19]:
from google.adk.agents import Agent
from google.adk.models.google_llm import Gemini

MODEL_NAME = "gemini-2.5-flash"

def multiply_numbers(a: int, b: int) -> dict:
    """
    Multiplies two integers.

    Args:
        a: First number.
        b: Second number.

    Returns:
        Dictionary containing the result.
    """

    return {
        "status": "success",
        "result": a * b
    }



root_agent = Agent(
    name="math_assistant",
    model=Gemini(model=MODEL_NAME),
    instruction="""
    You are a helpful math assistant.

    Whenever the user asks to multiply two numbers,
    use the multiply_numbers tool.
    """,
    tools=[multiply_numbers],
)


from google.adk.runners import InMemoryRunner
runner = InMemoryRunner(agent=root_agent)

response = await runner.run_debug(
    "what is the multipy of 2 and 2?",
    verbose=True,
)

math_assistant > [Calling tool: multiply_numbers({'b': 2, 'a': 2})]
math_assistant > [Tool result: {'status': 'success', 'result': 4}]
math_assistant > The product of 2 and 2 is 4.


In [20]:
response

[Event(model_version='gemini-2.5-flash', content=Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'a': 2,
           'b': 2
         },
         id='adk-d5ab0439-d7cb-4b2a-8ce2-770206243ffa',
         name='multiply_numbers'
       ),
       thought_signature=b'\n\xf1\x01\x01\x0c9\xd6\xc7J4\xef\xae\xa2J$e\x0e\xa0\xdb\xe3\x9c\x1f\xea\r&`g{ift\xd4\xd0\x0c\xcc\xeb\xc0\xae\xb8\xaeZ\xda\x7f98s%\xc7<&@W\xca\x1d\xc4\x1d\x0et\x83\xbd\x8a\xab\x0fe\xe5\xa9\xca\xe7\x0e0\x88\x06\xf3\xee\xdf\x85IkW\xea\x135d\xba\x05F\xfc\xb0\xe0w\xe5U\x13\x17\xe2?\x7f...'
     ),
   ],
   role='model'
 ), grounding_metadata=None, partial=None, turn_complete=None, turn_complete_reason=None, finish_reason=<FinishReason.STOP: 'STOP'>, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=GenerateContentResponseUsageMetadata(
   candidates_token_count=20,
   prompt_token_count=138,
   prompt_tokens_details=[
     ModalityTokenCount(
     

In [21]:
from google.adk.tools import AgentTool

# Define agent tool
tool_agent = Agent(
  model=MODEL_NAME,
  name="tool_agent",
  description="Returns the capital city for any country or state",
  instruction="""When the user gives you the name of a country (e.g. Germany),
  answer with the name of the capital city of that country.
  Otherwise, tell the user you are not able to help them."""
)

# Define primary agent
root_agent = Agent(
    name="assistant_with_agent_tool",
    model=Gemini(
        model=MODEL_NAME,
        retry_options=retry_config
        ),
  description="Answers user questions and gives advice",
  instruction="""Use the tools you have available to answer the user's questions""",
  tools=[AgentTool(agent=tool_agent)]
)


runner = InMemoryRunner(agent=root_agent)

response = await runner.run_debug(
    "I want to visit Germany. Which city do you recommend I visit first?",
    verbose=True,
)

assistant_with_agent_tool > [Calling tool: tool_agent({'request': 'Germany'})]
assistant_with_agent_tool > [Tool result: {'result': 'Berlin'}]
assistant_with_agent_tool > I recommend you visit Berlin first. It is the capital of Germany and has a rich history and vibrant culture.


In [22]:
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

# MCP integration with Everything Server
mcp_server = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="npx",  # Run MCP server via npx
            args=[
                "-y",  # Argument for npx to auto-confirm install
                "@modelcontextprotocol/server-everything",
            ],
            tool_filter=["getTinyImage"],
        ),
        timeout=30,
    )
)

c:\E_DRIVE\Langraph-Refresher\.venv\Lib\site-packages\google\adk\features\_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [23]:
# Create image agent with MCP integration
root_agent = Agent(
    name="assistant_with_mcp_tool",
    model=Gemini(
        model=MODEL_NAME,
        retry_options=retry_config
        ),
    instruction="Use the MCP Tool to generate images for user queries",
    tools=[mcp_server],
)

In [24]:

from google.adk.runners import InMemoryRunner
runner = InMemoryRunner(agent=root_agent)

response = await runner.run_debug(
    "generate image for flower?",
    verbose=True,
)

c:\E_DRIVE\Langraph-Refresher\.venv\Lib\site-packages\google\adk\tools\mcp_tool\mcp_toolset.py:304: UserWarning: [EXPERIMENTAL] feature FeatureName._MCP_GRACEFUL_ERROR_HANDLING is enabled.
  session = await self._mcp_session_manager.create_session(
Error on session runner task: fileno
Error on session runner task: fileno
Failed to get tools from toolset McpToolset: Failed to create MCP session: Failed to create MCP session: fileno
Error on session runner task: fileno
Error on session runner task: fileno
Failed to get tools from toolset McpToolset: Failed to create MCP session: Failed to create MCP session: fileno


In [25]:

from google.adk.plugins.logging_plugin import LoggingPlugin

root_agent = Agent(
    name="assistant",
    model=Gemini(
        model=MODEL_NAME,
        retry_options=retry_config
    ),
    description="A simple agent that can answer general questions.",
    instruction="""You are a helpful assistant.
    Use Google Search for current info or if unsure.""",
    tools=[google_search],
)

runner = InMemoryRunner(
    agent=root_agent,
    plugins=[
        LoggingPlugin()  # Handles standard Observability logging across ALL agents
    ],
)

response = await runner.run_debug("When was the Kaggle 5-Day AI Agents Intensive Course happening?")

[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-b529591c-1368-4849-82d9-4de5959d2da4
[logging_plugin]    Session ID: debug_session_id
[logging_plugin]    User ID: debug_user_id
[logging_plugin]    App Name: InMemoryRunner
[logging_plugin]    Root Agent: assistant
[logging_plugin]    User Content: text: 'When was the Kaggle 5-Day AI Agents Intensive Course happening?'
[logging_plugin] 🏃 INVOCATION STARTING
[logging_plugin]    Invocation ID: e-b529591c-1368-4849-82d9-4de5959d2da4
[logging_plugin]    Starting Agent: assistant
[logging_plugin] 🤖 AGENT STARTING
[logging_plugin]    Agent Name: assistant
[logging_plugin]    Invocation ID: e-b529591c-1368-4849-82d9-4de5959d2da4
[logging_plugin] 🧠 LLM REQUEST
[logging_plugin]    Model: gemini-2.5-flash
[logging_plugin]    Agent: assistant
[logging_plugin]    System Instruction: 'You are a helpful assistant.
    Use Google Search for current info or if unsure.

You are an agent. Your internal name is "assistant". Th